In [5]:
import os
import cv2
import numpy as np
from scipy.ndimage import gaussian_filter
from scipy.stats import spearmanr
from scipy.io import loadmat

In [3]:
def calculate_psnr(img_ref, img_dist, max_val=255.0):
    mse = np.mean((img_ref - img_dist) ** 2)
    if mse == 0:
        return float('inf')
    return 10 * np.log10((max_val ** 2) / mse)

def calculate_ssim_components(img_ref, img_dist, max_val=255.0, k1=0.01, k2=0.03):
    c1 = (k1 * max_val) ** 2
    c2 = (k2 * max_val) ** 2
    c3 = c2 / 2

    mu_x = gaussian_filter(img_ref, sigma=1.5, truncate=3.5)
    mu_y = gaussian_filter(img_dist, sigma=1.5, truncate=3.5)
    
    mu_x_sq = mu_x ** 2
    mu_y_sq = mu_y ** 2
    mu_xy = mu_x * mu_y

    # 3. Compute local variances and covariance (Contrast/Structure estimation)
    sigma_x_sq = gaussian_filter(img_ref ** 2, sigma=1.5, truncate=3.5) - mu_x_sq
    sigma_y_sq = gaussian_filter(img_dist ** 2, sigma=1.5, truncate=3.5) - mu_y_sq
    sigma_xy = gaussian_filter(img_ref * img_dist, sigma=1.5, truncate=3.5) - mu_xy

    # Handle tiny negative values caused by floating point inaccuracies
    sigma_x = np.sqrt(np.maximum(sigma_x_sq, 0))
    sigma_y = np.sqrt(np.maximum(sigma_y_sq, 0))

    # 4. Compute the independent components using local constants
    l_map = (2 * mu_xy + c1) / (mu_x_sq + mu_y_sq + c1)
    c_map = (2 * sigma_x * sigma_y + c2) / (sigma_x_sq + sigma_y_sq + c2)
    s_map = (sigma_xy + c3) / (sigma_x * sigma_y + c3)
    
    # 5. Compute overall SSIM map
    ssim_map = l_map * c_map * s_map

    # Return the spatial averages (Mean SSIM components)
    return np.mean(l_map), np.mean(c_map), np.mean(s_map), np.mean(ssim_map)

In [ ]:
gblur_dir = 'hw5/gblur'
ref_dir = 'hw5/refimgs'

In [6]:
mat_data = loadmat('hw5/hw5.mat')

In [29]:
raw_refnames = mat_data['refnames_blur'].flatten()
dmos_scores = mat_data['blur_dmos'].flatten()
org_indicators = mat_data['blur_orgs'].flatten()

ref_names = []
for item in raw_refnames:
    if isinstance(item, np.ndarray) and item.size > 0:
        ref_names.append(str(item[0]))
    else:
        ref_names.append(str(item))

In [30]:
psnr_scores = []
l_scores = []
c_scores = []
s_scores = []
ssim_scores = []
valid_dmos = []

In [31]:
for i in range(len(ref_names)):
    # 1 indicates an original image; skip these to isolate distorted images.
    if org_indicators[i] == 1:
        continue
    valid_dmos.append(dmos_scores[i])

    distorted_filename = f"img{i+1}.bmp" 
    ref_filename = ref_names[i]
    dist_path = os.path.join(gblur_dir, distorted_filename)
    ref_path = os.path.join(ref_dir, ref_filename)

    img_dist = cv2.imread(dist_path, cv2.IMREAD_GRAYSCALE).astype(np.float64)
    img_ref = cv2.imread(ref_path, cv2.IMREAD_GRAYSCALE).astype(np.float64)

    # Compute isolated metrics
    psnr_val = calculate_psnr(img_ref, img_dist)
    psnr_scores.append(psnr_val)

    l_val, c_val, s_val, ssim_val = calculate_ssim_components(img_ref, img_dist)
    l_scores.append(l_val)
    c_scores.append(c_val)
    s_scores.append(s_val)
    ssim_scores.append(ssim_val)

In [32]:
valid_dmos = np.array(valid_dmos)

srocc_psnr, _ = spearmanr(psnr_scores, valid_dmos)
srocc_l, _ = spearmanr(l_scores, valid_dmos)
srocc_c, _ = spearmanr(c_scores, valid_dmos)
srocc_s, _ = spearmanr(s_scores, valid_dmos)
srocc_ssim, _ = spearmanr(ssim_scores, valid_dmos)

In [33]:
print(f"1. PSNR:               {abs(srocc_psnr):.4f}")
print(f"2. SSIM (Luminance):   {abs(srocc_l):.4f}")
print(f"3. SSIM (Contrast):    {abs(srocc_c):.4f}")
print(f"4. SSIM (Structure):   {abs(srocc_s):.4f}")
print(f"5. Overall SSIM:       {abs(srocc_ssim):.4f}")

1. PSNR:               0.7823
2. SSIM (Luminance):   0.9335
3. SSIM (Contrast):    0.9068
4. SSIM (Structure):   0.9055
5. Overall SSIM:       0.9036
